# DWD Weather → Bridge-Level Feature Engineering — FINAL WORKFLOW

**Goal:** Assign every eligible bridge to an appropriate DWD weather station, control spatial distance and mapping quality, process daily weather observations, and create weather features at **bridge-year level**.

**Bridge population rule:** only bridges with `Baujahr > 1900` are eligible.

### Pipeline
`Fragestellung → DWD Source → Raw Station → Clean/Core → Spatial Station → Bridge Population → Nearest Station Mapping → Distance/Quality Control → Weather File Inventory → Cached Daily Data → Cleaning/Validation → Station-Year Features → Bridge-Year Features → Visualization → Final Results`

**No prediction / supervised ML is performed in this notebook.**

# DATA TRANSFER / INPUT–OUTPUT MANIFEST — Notebook 03

## Portable location rule
All local paths are derived from one `PROJECT_ROOT`. On another computer:
1. put `Dataset_PlanA-B` under the project root and start Jupyter from that root, **or**
2. set `BRIDGE_PROJECT_ROOT` to the project root.

Required local Weather structure:

```text
<PROJECT_ROOT>/
├── Dataset_PlanA-B/
│   └── Weather/
│       ├── raw/
│       ├── processed/
│       ├── weather_daily.parquet
│       └── dwd_weather_station_annual.parquet
└── Output_PlanA-B/
```

## Data inventory

| Class | Data | How obtained/read | Role |
|---|---|---|---|
| External URL | DWD station metadata | `DWD_STATION_METADATA_URL` via `urlopen()` | Station master data |
| External URL | DWD historical archive | `DWD_HISTORICAL_URL` | Daily historical source |
| External URL | DWD recent archive | `DWD_RECENT_URL` | Daily recent source |
| Local raw cache | DWD station metadata + ZIP archives | `Dataset_PlanA-B/Weather/dwd_weather_raw/` | Reusable source/cache |
| Local processed cache | Per-archive Parquet files | `Dataset_PlanA-B/Weather/processed/` | Avoids reprocessing ZIPs |
| Local intermediate | `weather_daily.parquet` | Written/read inside Notebook 03 | Compact daily weather layer |
| Local intermediate | `dwd_weather_station_annual.parquet` | Written inside Notebook 03 | Station-year aggregation |
| PostgreSQL input | `final.bridge` | SQL | Bridge population/spatial integration |
| PostgreSQL output | raw/cleaned/transformed Weather layers | SQL | Processing layers |
| PostgreSQL output | `final.weather` | SQL | **Canonical downstream Weather output** |

## Downstream rule

Notebook 04 consumes **`final.weather`**. It does not need to read DWD raw ZIPs, `weather_daily.parquet`, or `dwd_weather_station_annual.parquet` directly.

The local Weather files are source/cache/intermediate artifacts. `final.weather` is the canonical interface to the next notebook.

## Transfer to another computer

For offline reproducibility, transfer the complete `Dataset_PlanA-B/Weather/` directory and the PostgreSQL database/export. With internet access, the DWD raw/cache layers can be rebuilt from the documented URLs.


## 1. Project Setup

All database settings, paths and quality thresholds are centralized here.

In [1]:

from getpass import getpass
import os, re, pandas as pd
from pathlib import Path
import zipfile
import ssl
from urllib.parse import urljoin
from urllib.request import Request, urlopen

try:
    import certifi
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "certifi"])
    import certifi

DWD_SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

PG_HOST="localhost"
PG_PORT=5432
PG_DATABASE="Final_Project"
PG_USER="postgres"

BRIDGE_TABLE="final.bridge"
WEATHER_STATION_RAW="raw_bridge_statistics.dwd_weather_stations_raw"
WEATHER_STATION_CLEAN="cleaned.dwd_weather_stations_clean"
WEATHER_STATION_CORE="cleaned.dwd_weather_stations_core"
WEATHER_STATION_GEO="transformed.dwd_weather_station_geo"
BRIDGE_WEATHER_MAP="transformed.bridge_weather_station"
STATION_ANNUAL="transformed.dwd_weather_station_annual"
BRIDGE_WEATHER_ANNUAL="transformed.bridge_weather_annual"
FINAL_WEATHER_TABLE="final.weather"

MIN_BUILD_YEAR=1900
MIN_WEATHER_YEAR=1901
WEATHER_MAX_DISTANCE_KM=50.0

DWD_STATION_METADATA_URL=(
    "https://opendata.dwd.de/climate_environment/CDC/"
    "observations_germany/climate/daily/kl/"
    "historical/KL_Tageswerte_Beschreibung_Stationen.txt"
)
DWD_HISTORICAL_URL=(
    "https://opendata.dwd.de/climate_environment/CDC/"
    "observations_germany/climate/daily/kl/historical/"
)
DWD_RECENT_URL=(
    "https://opendata.dwd.de/climate_environment/CDC/"
    "observations_germany/climate/daily/kl/recent/"
)


import os
from pathlib import Path

def find_project_root():
    env_root = os.environ.get("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT is set, but Dataset_PlanA-B was not found under: {root}"
        )

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Start Jupyter from the project root "
        "or set BRIDGE_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)

WEATHER_DIR = DATASET_ROOT / "Weather"
weather_raw_dir = WEATHER_DIR / "dwd_weather_raw"
weather_parquet_dir = WEATHER_DIR / "processed"
weather_daily_path = WEATHER_DIR / "weather_daily.parquet"
weather_station_annual_path = WEATHER_DIR / "dwd_weather_station_annual.parquet"

weather_parquet_dir.mkdir(parents=True, exist_ok=True)
weather_raw_dir.mkdir(parents=True, exist_ok=True)

DWD_RAW_FILE = weather_raw_dir / "KL_Tageswerte_Beschreibung_Stationen.txt"
DWD_PARSED_RAW_FILE = weather_parquet_dir / "dwd_weather_stations_raw.csv"
DWD_CLEAN_FILE = weather_parquet_dir / "dwd_weather_stations_clean.csv"
DWD_CORE_FILE = weather_parquet_dir / "dwd_weather_stations_core.csv"

PG_PASSWORD=getpass("Enter PostgreSQL password: ")
engine=create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
)
with engine.connect() as conn:
    print("Connected:", conn.execute(text("SELECT current_database()")).scalar())

print("Weather directory:", WEATHER_DIR)
print("Weather raw cache:", weather_raw_dir)
print("Weather processed cache:", weather_parquet_dir)


Project root: C:\Datenanalyse\final Project
Dataset root: C:\Datenanalyse\final Project\Dataset_PlanA-B
Output root: C:\Datenanalyse\final Project\Output_PlanA-B
Connected: Final_Project
Weather directory: C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather
Weather raw cache: C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather\dwd_weather_raw
Weather processed cache: C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather\processed


### HTTPS / SSL note

DWD downloads use certificate verification through the `certifi` CA bundle. The notebook does **not** disable SSL verification. If `certifi` is missing, it is installed automatically into the active Python environment.


## 2. DWD Station Metadata — Raw → Clean → Core

Retrieve the official DWD station metadata, preserve the source, parse it, validate it, and persist the reusable station layers.

In [2]:
request = Request(
    DWD_STATION_METADATA_URL,
    headers={"User-Agent": "Mozilla/5.0"}
)
with urlopen(request, timeout=60, context=DWD_SSL_CONTEXT) as response:
    metadata_bytes = response.read()

print("DWD metadata download successful.")
print("Bytes downloaded:", f"{len(metadata_bytes):,}")

DWD metadata download successful.
Bytes downloaded: 1,388,997


In [3]:
DWD_RAW_FILE.parent.mkdir(parents=True, exist_ok=True)
DWD_RAW_FILE.write_bytes(metadata_bytes)

print("Original DWD file saved:")
print(DWD_RAW_FILE)
print("Size:", f"{DWD_RAW_FILE.stat().st_size:,}", "bytes")

Original DWD file saved:
C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather\dwd_weather_raw\KL_Tageswerte_Beschreibung_Stationen.txt
Size: 1,388,997 bytes


In [4]:
metadata_text = metadata_bytes.decode(
    "latin1",
    errors="replace"
)

metadata_lines = metadata_text.splitlines()

print("Number of lines:", f"{len(metadata_lines):,}")
print()
print("First 50 lines:")
for i, line in enumerate(metadata_lines[:50], start=1):
    print(f"{i:02d}: {line}")

Number of lines: 1,388

First 50 lines:
01: Stations_id von_datum bis_datum Stationshoehe geoBreite geoLaenge Stationsname Bundesland Abgabe
02: ----------- --------- --------- ------------- --------- --------- ----------------------------------------- ---------- ------
03: 00001 19370101 19860630            478     47.8413    8.8493 Aach                                     Baden-Württemberg                        Frei                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [5]:
station_pattern = re.compile(
    r"^(?P<station_id>.{5})\s"
    r"(?P<start_date>\d{8})\s"
    r"(?P<end_date>\d{8})\s"
    r"(?P<elevation_m>.{14})\s"
    r"(?P<latitude>.{11})\s"
    r"(?P<longitude>.{9})\s"
    r"(?P<station_name>.{40})\s"
    r"(?P<federal_state>.+)$"
)

records = []

for line in metadata_lines:
    match = station_pattern.match(line)
    if match:
        records.append(match.groupdict())

dwd_raw_df = pd.DataFrame(records)

print("DWD station records parsed.")
print("Rows:", f"{len(dwd_raw_df):,}")
print("Columns:", len(dwd_raw_df.columns))
display(dwd_raw_df.head(10))

DWD station records parsed.
Rows: 1,386
Columns: 8


,station_id,start_date,end_date,elevation_m,latitude,longitude,station_name,federal_state
0,00001,19370101,19860630,478,47.8413,8.8493,Aach,Baden-Württemberg Frei ...
1,00003,18910101,20110331,202,50.7827,6.0941,Aachen,Nordrhein-Westfalen Frei ...
2,00011,19800901,20260918,680,47.9736,8.5205,Donaueschingen (Landeplatz),Baden-Württemberg Frei ...
3,00044,19690101,20260918,44,52.9336,8.2370,Großenkneten,Niedersachsen Frei ...
4,00052,19690101,20011231,46,53.6623,10.1990,Ahrensburg-Wulfsdorf,Schleswig-Holstein Frei ...
5,00061,19750701,19780831,339,48.8443,12.6171,Aiterhofen,Bayern Frei ...
6,00070,19730601,19860930,712,48.2052,9.0371,Albstadt-Ebingen,Baden-Württemberg Frei ...
7,00071,19861101,20191231,759,48.2156,8.9784,Albstadt-Badkap,Baden-Württemberg Frei ...
8,00072,19780901,19950531,794,48.2766,9.0001,Albstadt-Onstmettingen,Baden-Württemberg Frei ...
9,00073,19590301,20260918,374,48.6183,13.0620,Aldersbach-Kramersepp,Bayern Frei ...


In [6]:
required_columns = [
    "station_id",
    "start_date",
    "end_date",
    "elevation_m",
    "latitude",
    "longitude",
    "station_name",
    "federal_state",
]

missing_columns = [
    c for c in required_columns
    if c not in dwd_raw_df.columns
]

if missing_columns:
    raise RuntimeError(
        "Required DWD columns are missing: "
        + ", ".join(missing_columns)
    )

duplicate_station_ids = int(
    dwd_raw_df["station_id"]
    .astype(str)
    .str.strip()
    .duplicated()
    .sum()
)

print("Required-column check: PASS")
print("Duplicate station IDs:", duplicate_station_ids)
print("Rows:", f"{len(dwd_raw_df):,}")

if duplicate_station_ids:
    print("Duplicate station IDs detected; the cleaned layer will retain the first record per station.")

Required-column check: PASS
Duplicate station IDs: 0
Rows: 1,386


In [7]:
RAW_TABLE = "raw_bridge_statistics.dwd_weather_stations_raw"

raw_upload_df = dwd_raw_df.copy()
raw_upload_df.insert(0, "source_file", DWD_RAW_FILE.name)

with engine.begin() as conn:
    conn.execute(
        text("CREATE SCHEMA IF NOT EXISTS raw_bridge_statistics")
    )

raw_upload_df.to_sql(
    "dwd_weather_stations_raw",
    engine,
    schema="raw_bridge_statistics",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi",
)

raw_upload_df.to_csv(
    DWD_PARSED_RAW_FILE,
    index=False,
    encoding="utf-8"
)

print("Raw weather table created:", RAW_TABLE)
print("Rows:", f"{len(raw_upload_df):,}")
print("Parsed raw CSV:", DWD_PARSED_RAW_FILE)

Raw weather table created: raw_bridge_statistics.dwd_weather_stations_raw
Rows: 1,386
Parsed raw CSV: C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather\processed\dwd_weather_stations_raw.csv


In [8]:
CLEAN_TABLE = "cleaned.dwd_weather_stations_clean"

clean_df = raw_upload_df.copy()

for column in [
    "station_id",
    "station_name",
    "federal_state",
]:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
    )

for column in [
    "elevation_m",
    "latitude",
    "longitude",
]:
    clean_df[column] = pd.to_numeric(
        clean_df[column].astype("string").str.strip(),
        errors="coerce"
    )

for column in [
    "start_date",
    "end_date",
]:
    clean_df[column] = pd.to_datetime(
        clean_df[column].astype("string").str.strip(),
        format="%Y%m%d",
        errors="coerce"
    )

valid_station_id = (
    clean_df["station_id"].notna()
    & clean_df["station_id"].astype("string").str.strip().ne("")
)

valid_coordinates = (
    clean_df["latitude"].between(-90, 90)
    & clean_df["longitude"].between(-180, 180)
)

clean_df = clean_df.loc[
    valid_station_id & valid_coordinates
].copy()

clean_df = clean_df.drop_duplicates(
    subset=["station_id"],
    keep="first"
).reset_index(drop=True)

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS cleaned"))

clean_df.to_sql(
    "dwd_weather_stations_clean",
    engine,
    schema="cleaned",
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi",
)

clean_df.to_csv(
    DWD_CLEAN_FILE,
    index=False,
    encoding="utf-8"
)

print("Cleaned weather table created:", CLEAN_TABLE)
print("Rows:", f"{len(clean_df):,}")
print("Columns:", len(clean_df.columns))
print("Clean CSV:", DWD_CLEAN_FILE)

Cleaned weather table created: cleaned.dwd_weather_stations_clean
Rows: 1,386
Columns: 9
Clean CSV: C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather\processed\dwd_weather_stations_clean.csv


In [9]:
clean_validation = pd.read_sql(
    text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT station_id) AS unique_stations,
            COUNT(latitude) AS latitude_available,
            COUNT(longitude) AS longitude_available,
            COUNT(*) FILTER (
                WHERE latitude BETWEEN -90 AND 90
                  AND longitude BETWEEN -180 AND 180
            ) AS valid_coordinates,
            COUNT(*) FILTER (
                WHERE latitude IS NULL
                   OR longitude IS NULL
            ) AS missing_coordinates,
            MIN(latitude) AS min_latitude,
            MAX(latitude) AS max_latitude,
            MIN(longitude) AS min_longitude,
            MAX(longitude) AS max_longitude
        FROM cleaned.dwd_weather_stations_clean
    """),
    engine
)

display(clean_validation)

v = clean_validation.iloc[0]

assert int(v["total_rows"]) == int(v["unique_stations"])
assert int(v["valid_coordinates"]) == int(v["total_rows"])
assert int(v["missing_coordinates"]) == 0

print("Cleaned station validation: PASS")

,total_rows,unique_stations,latitude_available,longitude_available,valid_coordinates,missing_coordinates,min_latitude,max_latitude,min_longitude,max_longitude
0,1386,1386,1386,1386,1386,0,47.3984,55.011,6.0244,14.9506


Cleaned station validation: PASS


In [10]:
CORE_TABLE = "cleaned.dwd_weather_stations_core"

core_columns = [
    "station_id",
    "start_date",
    "end_date",
    "elevation_m",
    "latitude",
    "longitude",
    "station_name",
    "federal_state",
]

quoted_core = ", ".join(
    '"' + c.replace('"', '""') + '"'
    for c in core_columns
)

with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {CORE_TABLE}"))
    conn.execute(
        text(
            f"CREATE TABLE {CORE_TABLE} AS "
            f"SELECT {quoted_core} "
            f"FROM cleaned.dwd_weather_stations_clean"
        )
    )

dwd_core_df = pd.read_sql(
    text(f"SELECT * FROM {CORE_TABLE}"),
    engine
)

dwd_core_df.to_csv(
    DWD_CORE_FILE,
    index=False,
    encoding="utf-8"
)

print("Core weather-station table created:", CORE_TABLE)
print("Rows:", f"{len(dwd_core_df):,}")
print("Columns:", len(core_columns))
print("Core CSV:", DWD_CORE_FILE)

Core weather-station table created: cleaned.dwd_weather_stations_core
Rows: 1,386
Columns: 8
Core CSV: C:\Datenanalyse\final Project\Dataset_PlanA-B\Weather\processed\dwd_weather_stations_core.csv


## 3. DWD Station Geometry

Create and validate the PostGIS station geometry layer.

In [11]:
TRANSFORMED_TABLE = "transformed.dwd_weather_station_geo"

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS transformed"))
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))

    conn.execute(
        text(f"DROP TABLE IF EXISTS {TRANSFORMED_TABLE}")
    )

    conn.execute(
        text(f"""
            CREATE TABLE {TRANSFORMED_TABLE} AS
            SELECT
                *,
                ST_SetSRID(
                    ST_MakePoint(longitude, latitude),
                    4326
                ) AS geom
            FROM {CORE_TABLE}
            WHERE longitude BETWEEN -180 AND 180
              AND latitude BETWEEN -90 AND 90
        """)
    )

    conn.execute(
        text(
            f"CREATE INDEX dwd_weather_station_geo_geom_idx "
            f"ON {TRANSFORMED_TABLE} USING GIST (geom)"
        )
    )

print("Spatial weather table created:", TRANSFORMED_TABLE)

Spatial weather table created: transformed.dwd_weather_station_geo


In [12]:
geo_check = pd.read_sql(
    text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(geom) AS geometry_rows,
            COUNT(*) FILTER (
                WHERE geom IS NOT NULL
                  AND ST_SRID(geom) = 4326
                  AND GeometryType(geom) = 'POINT'
            ) AS valid_point_rows,
            MIN(latitude) AS min_latitude,
            MAX(latitude) AS max_latitude,
            MIN(longitude) AS min_longitude,
            MAX(longitude) AS max_longitude
        FROM transformed.dwd_weather_station_geo
    """),
    engine
)

display(geo_check)

g = geo_check.iloc[0]

assert int(g["total_rows"]) == len(dwd_core_df)
assert int(g["geometry_rows"]) == len(dwd_core_df)
assert int(g["valid_point_rows"]) == len(dwd_core_df)

index_check = pd.read_sql(
    text("""
        SELECT indexname, indexdef
        FROM pg_indexes
        WHERE schemaname = 'transformed'
          AND tablename = 'dwd_weather_station_geo'
          AND indexname = 'dwd_weather_station_geo_geom_idx'
    """),
    engine
)

display(index_check)

assert len(index_check) == 1

print("PostGIS geometry validation: PASS")
print("Spatial index validation: PASS")

,total_rows,geometry_rows,valid_point_rows,min_latitude,max_latitude,min_longitude,max_longitude
0,1386,1386,1386,47.3984,55.011,6.0244,14.9506


,indexname,indexdef
0,dwd_weather_station_geo_geom_idx,CREATE INDEX dwd_weather_station_geo_geom_idx ...


PostGIS geometry validation: PASS
Spatial index validation: PASS


In [13]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_dwd_weather_station_geo_geom
        ON transformed.dwd_weather_station_geo
        USING GIST (geom);
    """))

print("DWD station spatial index is ready.")

DWD station spatial index is ready.


## 4. Bridge Population — Strictly `Baujahr > 1900`

The previous notebook had a separate ML bridge population using `>= 1900`. That redundant layer is removed. This workflow uses the final bridge master directly and applies the project rule **`Baujahr > 1900`**.

In [14]:
with engine.connect() as conn:
    bridge_exists = conn.execute(
        text("SELECT to_regclass(:table_name)"),
        {"table_name": BRIDGE_TABLE}
    ).scalar()

if bridge_exists is None:
    raise RuntimeError(
        f"{BRIDGE_TABLE} does not exist. Run BASt-GIS_FINAL.ipynb first to create final.bridge."
    )

# final.bridge stores the ArcGIS Web-Mercator coordinates as geom_x / geom_y.
# Reconstruct a WGS84 PostGIS point for spatial matching.
bridge_population = pd.read_sql(
    text(f"""
        SELECT
            b.*,
            ST_Transform(
                ST_SetSRID(
                    ST_MakePoint(b.geom_x, b.geom_y),
                    3857
                ),
                4326
            ) AS geom
        FROM {BRIDGE_TABLE} AS b
        WHERE b.baujahr > :min_year
          AND b.baujahr IS NOT NULL
          AND b.geom_x IS NOT NULL
          AND b.geom_y IS NOT NULL
        ORDER BY b.id_nr
    """),
    engine,
    params={"min_year": MIN_BUILD_YEAR}
)

required = ["id_nr", "baujahr", "geom_x", "geom_y", "geom"]
missing = [c for c in required if c not in bridge_population.columns]
if missing:
    raise KeyError(f"Required bridge columns missing: {missing}")

assert bridge_population["baujahr"].gt(MIN_BUILD_YEAR).all()
assert bridge_population["id_nr"].notna().all()
assert bridge_population["id_nr"].is_unique

print("Eligible bridges with valid coordinates:", f"{len(bridge_population):,}")
print(
    "Baujahr:",
    int(bridge_population.baujahr.min()),
    "-",
    int(bridge_population.baujahr.max())
)
print("PASS: only bridges with Baujahr > 1900 and valid GIS coordinates enter the weather pipeline.")


Eligible bridges with valid coordinates: 51,428
Baujahr: 1901 - 3019
PASS: only bridges with Baujahr > 1900 and valid GIS coordinates enter the weather pipeline.


## 5. DWD Weather File Inventory

Check historical and recent DWD files for the stations actually used by bridges.

In [15]:
request = Request(DWD_HISTORICAL_URL, headers={"User-Agent": "Mozilla/5.0"})
with urlopen(request, timeout=60, context=DWD_SSL_CONTEXT) as response:
    historical_html = response.read().decode("utf-8", errors="replace")
hist_matches=re.findall(
    r'href="(tageswerte_KL_(\d{5})_(\d{8})_(\d{8})_hist\.zip)"',
    historical_html
)
dwd_inventory=pd.DataFrame(
    hist_matches,columns=["filename","station_id","start_raw","end_raw"]
)
if not dwd_inventory.empty:
    dwd_inventory["start_date"]=pd.to_datetime(dwd_inventory.start_raw,format="%Y%m%d")
    dwd_inventory["end_date"]=pd.to_datetime(dwd_inventory.end_raw,format="%Y%m%d")
    dwd_inventory["url"]=dwd_inventory.filename.map(lambda x:urljoin(DWD_HISTORICAL_URL,x))

historical_ids=set(dwd_inventory.station_id) if not dwd_inventory.empty else set()
request = Request(DWD_RECENT_URL, headers={"User-Agent": "Mozilla/5.0"})
with urlopen(request, timeout=60, context=DWD_SSL_CONTEXT) as response:
    recent_html = response.read().decode("utf-8", errors="replace")
recent_matches=re.findall(r'href="(tageswerte_KL_(\d{5})_akt\.zip)"',recent_html)
recent_inventory=pd.DataFrame(recent_matches,columns=["filename","station_id"])
recent_ids=set(recent_inventory.station_id) if not recent_inventory.empty else set()
inventory_summary=pd.DataFrame({
    "metric":[
        "historical_stations","recent_stations","stations_with_either_archive"
    ],
    "value":[
        len(historical_ids),
        len(recent_ids),
        len(historical_ids | recent_ids)
    ]
})
display(inventory_summary)

# Persist the archive availability used as the candidate-station QC gate.
archive_ids = sorted(historical_ids | recent_ids)
archive_inventory = pd.DataFrame({
    "station_id": archive_ids,
    "has_historical": [sid in historical_ids for sid in archive_ids],
    "has_recent": [sid in recent_ids for sid in archive_ids],
})

archive_inventory.to_sql(
    "dwd_weather_station_archive_inventory",
    engine,
    schema="transformed",
    if_exists="replace",
    index=False,
)

assert archive_inventory.station_id.is_unique
assert archive_inventory["has_historical"].fillna(False).astype(bool).any()
assert archive_inventory["has_recent"].fillna(False).astype(bool).any()

print(
    "Archive-eligible DWD stations:",
    f"{len(archive_inventory):,}"
)


,metric,value
0,historical_stations,1285
1,recent_stations,574
2,stations_with_either_archive,1287


Archive-eligible DWD stations: 1,287


## 6. Bridge → Nearest DWD Station

Assign one nearest **archive-eligible** DWD station to every eligible bridge. A station is eligible only if it has a current DWD historical or recent daily archive. This prevents metadata-only stations from being selected and later failing the archive QC gate.


In [16]:
with engine.begin() as conn:
    conn.execute(text(f"""
        DROP TABLE IF EXISTS {BRIDGE_WEATHER_MAP};

        CREATE TABLE {BRIDGE_WEATHER_MAP} AS
        WITH bridge_geo AS (
            SELECT
                b.*,
                ST_Transform(
                    ST_SetSRID(
                        ST_MakePoint(b.geom_x, b.geom_y),
                        3857
                    ),
                    4326
                ) AS bridge_geom
            FROM {BRIDGE_TABLE} AS b
            WHERE b.baujahr > {MIN_BUILD_YEAR}
              AND b.baujahr IS NOT NULL
              AND b.geom_x IS NOT NULL
              AND b.geom_y IS NOT NULL
        )
        SELECT
            b.id_nr,
            b.bwnr,
            b.tbwnr,
            b.baujahr,
            ST_Y(b.bridge_geom) AS bridge_latitude,
            ST_X(b.bridge_geom) AS bridge_longitude,
            'DWD' AS weather_source,
            s.station_id AS weather_station_id,
            s.station_name,
            s.latitude AS station_latitude,
            s.longitude AS station_longitude,
            s.elevation_m AS station_elevation_m,
            s.federal_state AS station_federal_state,
            ST_Distance(
                b.bridge_geom::geography,
                s.geom::geography
            ) / 1000.0 AS weather_distance_km
        FROM bridge_geo AS b
        CROSS JOIN LATERAL (
            SELECT
                s.station_id,
                s.station_name,
                s.latitude,
                s.longitude,
                s.elevation_m,
                s.federal_state,
                s.geom
            FROM {WEATHER_STATION_GEO} AS s
            INNER JOIN transformed.dwd_weather_station_archive_inventory AS a
                ON a.station_id = s.station_id
               AND (a.has_historical OR a.has_recent)
            WHERE s.geom IS NOT NULL
            ORDER BY s.geom <-> b.bridge_geom
            LIMIT 1
        ) AS s;
    """))

    conn.execute(text(f"""
        CREATE INDEX IF NOT EXISTS idx_bridge_weather_station_station
        ON {BRIDGE_WEATHER_MAP}(weather_station_id)
    """))

    conn.execute(text(f"""
        CREATE INDEX IF NOT EXISTS idx_bridge_weather_station_distance
        ON {BRIDGE_WEATHER_MAP}(weather_distance_km)
    """))

mapping = pd.read_sql(
    text(f"SELECT * FROM {BRIDGE_WEATHER_MAP}"),
    engine
)

print("Assignments:", f"{len(mapping):,}")
print("Unique bridges:", f"{mapping.id_nr.nunique():,}")
print("Unique DWD stations:", f"{mapping.weather_station_id.nunique():,}")
print("PASS: nearest-station candidates are restricted to DWD stations with historical or recent archives.")


Assignments: 51,428
Unique bridges: 51,428
Unique DWD stations: 1,206
PASS: nearest-station candidates are restricted to DWD stations with historical or recent archives.


## 7. Mapping Distance & Quality Control

Distance is explicitly checked and classified. The raw distance remains in the output so downstream modelling can make an informed decision.

In [17]:
mapping["weather_quality"]=pd.cut(
    mapping["weather_distance_km"],
    bins=[-float("inf"),5,10,30,WEATHER_MAX_DISTANCE_KM,float("inf")],
    labels=["excellent","good","acceptable","borderline","too_far"],
    right=False
)

distance_summary=(
    mapping.groupby("weather_quality",observed=False)
    .size().reset_index(name="bridge_count")
)
distance_summary["percentage"]=(distance_summary.bridge_count/len(mapping)*100).round(2)
display(distance_summary)

display(mapping["weather_distance_km"].describe(
    percentiles=[.50,.75,.90,.95,.99]
).to_frame("weather_distance_km"))

checks=pd.DataFrame({
    "check":[
        "bridge_count","unique_bridge_ids","missing_station_id",
        "negative_distance",f"distance_over_{WEATHER_MAX_DISTANCE_KM:g}_km",
        "all_baujahr_gt_1900"
    ],
    "value":[
        len(mapping),mapping.id_nr.nunique(),
        mapping.weather_station_id.isna().sum(),
        (mapping.weather_distance_km<0).sum(),
        (mapping.weather_distance_km>WEATHER_MAX_DISTANCE_KM).sum(),
        mapping.baujahr.gt(MIN_BUILD_YEAR).all()
    ]
})
display(checks)

assert mapping.id_nr.is_unique
assert mapping.weather_station_id.notna().all()
assert mapping.weather_source.eq("DWD").all()
assert mapping.weather_distance_km.notna().all()
assert mapping.weather_distance_km.ge(0).all()
assert mapping.baujahr.gt(MIN_BUILD_YEAR).all()

archive_check = pd.read_sql(
    text(f"""
        SELECT COUNT(*) AS non_archive_mapped
        FROM {BRIDGE_WEATHER_MAP} m
        LEFT JOIN transformed.dwd_weather_station_archive_inventory a
          ON a.station_id = m.weather_station_id
        WHERE a.station_id IS NULL
           OR NOT (a.has_historical OR a.has_recent)
    """),
    engine
)
assert int(archive_check.iloc[0]["non_archive_mapped"]) == 0

# Distances above 50 km are retained for auditability but are flagged as "too_far".
# The final layer does not silently convert them to missing values.

mapping["weather_quality"]=mapping["weather_quality"].astype("string")
mapping.to_sql(
    "bridge_weather_station",engine,schema="transformed",
    if_exists="replace",index=False
)
print("PASS: one station per bridge and spatial quality is controlled.")

,weather_quality,bridge_count,percentage
0,excellent,17103,33.26
1,good,20354,39.58
2,acceptable,13944,27.11
3,borderline,27,0.05
4,too_far,0,0.00


,weather_distance_km
count,51428.000000
mean,7.648819
std,4.674185
min,0.048129
50%,6.840165
75%,10.383491
90%,13.872492
95%,16.336420
99%,21.825913
max,33.458766


,check,value
0,bridge_count,51428
1,unique_bridge_ids,51428
2,missing_station_id,0
3,negative_distance,0
4,distance_over_50_km,0
5,all_baujahr_gt_1900,True


PASS: one station per bridge and spatial quality is controlled.


## 8. Daily Weather Processing — Cached Parquet

Use the local immutable DWD ZIP archives and create a resumable Parquet cache. Existing Parquet files are not reprocessed.

In [18]:
zip_files=sorted(weather_raw_dir.glob("*.zip"))
if not zip_files:
    raise FileNotFoundError(f"No DWD ZIP archives found in {weather_raw_dir}")

DWD_REQUIRED_COLUMNS=["STATIONS_ID","MESS_DATUM","RSK","TXK","TNK"]

def parse_dwd_daily_zip(zip_path):
    source_type="historical" if zip_path.name.endswith("_hist.zip") else "recent"
    with zipfile.ZipFile(zip_path) as archive:
        observation_files=[
            n for n in archive.namelist()
            if n.startswith("produkt_klima_tag_") and n.endswith(".txt")
        ]
        if len(observation_files)!=1:
            raise ValueError(
                f"Expected one observation file in {zip_path.name}; found {len(observation_files)}"
            )
        with archive.open(observation_files[0]) as f:
            header=pd.read_csv(f,sep=";",encoding="latin1",nrows=0)
        normalized=[str(c).strip() for c in header.columns]
        positions=[i for i,c in enumerate(normalized) if c in DWD_REQUIRED_COLUMNS]
        missing=[c for c in DWD_REQUIRED_COLUMNS if c not in normalized]
        if missing:
            raise ValueError(f"{zip_path.name} missing required columns: {missing}")
        with archive.open(observation_files[0]) as f:
            daily=pd.read_csv(
                f,sep=";",encoding="latin1",usecols=positions,
                na_values=[-999,-999.0,"-999","-999.0"]
            )

    daily.columns=[str(c).strip() for c in daily.columns]
    daily=daily[DWD_REQUIRED_COLUMNS].copy()
    daily=daily.rename(columns={
        "STATIONS_ID":"station_id","MESS_DATUM":"observation_date",
        "RSK":"precipitation_mm","TXK":"temperature_max_c",
        "TNK":"temperature_min_c"
    })
    daily["station_id"]=(
        pd.to_numeric(daily.station_id,errors="coerce")
        .astype("Int64").astype("string").str.zfill(5)
    )
    daily["observation_date"]=pd.to_datetime(
        daily.observation_date.astype("string").str.strip(),
        format="%Y%m%d",errors="coerce"
    )
    for c in ["precipitation_mm","temperature_max_c","temperature_min_c"]:
        daily[c]=pd.to_numeric(daily[c],errors="coerce")

    daily=daily.dropna(subset=["station_id","observation_date"])
    daily["precipitation_mm"]=daily.precipitation_mm.where(daily.precipitation_mm>=0)
    daily["temperature_max_c"]=daily.temperature_max_c.where(
        daily.temperature_max_c.between(-50,60)
    )
    daily["temperature_min_c"]=daily.temperature_min_c.where(
        daily.temperature_min_c.between(-60,50)
    )
    daily["temperature_mean_c"] = (
    daily[["temperature_min_c", "temperature_max_c"]]
    .mean(axis=1, skipna=False)
)
    daily=(
        daily.sort_values("observation_date")
        .drop_duplicates(["station_id","observation_date"],keep="last")
        .reset_index(drop=True)
    )
    daily["source_type"]=source_type
    daily["source_file"]=zip_path.name
    return daily

processed_log=[]
for zip_path in zip_files:
    parquet_path=weather_parquet_dir/f"{zip_path.stem}.parquet"
    if parquet_path.exists() and parquet_path.stat().st_size>0:
        processed_log.append({"source_file":zip_path.name,"status":"already_processed"})
        continue
    daily=parse_dwd_daily_zip(zip_path)
    daily.to_parquet(parquet_path,index=False,compression="snappy")
    processed_log.append({"source_file":zip_path.name,"status":"processed"})

processed_log=pd.DataFrame(processed_log)
display(processed_log.status.value_counts().to_frame("count"))
print("Cached archives:",len(processed_log))

,count
status,
processed,1200


Cached archives: 1200


## 9. Daily Weather Layer — Cleaning & Validation

Consolidate the Parquet cache, correct the three identified temperature-order anomalies, and validate the daily layer.

In [19]:
parquet_files=sorted(weather_parquet_dir.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No processed Parquet files found in {weather_parquet_dir}")

weather_daily=pd.concat(
    (pd.read_parquet(p) for p in parquet_files),ignore_index=True
)
weather_daily=(
    weather_daily.sort_values(
        ["station_id","observation_date","source_type"]
    )
    .drop_duplicates(["station_id","observation_date"],keep="first")
    .reset_index(drop=True)
)

FINAL_WEATHER_COLUMNS=[
    "station_id","observation_date",
    "temperature_min_c","temperature_max_c",
    "temperature_mean_c","precipitation_mm"
]
weather_daily=weather_daily[FINAL_WEATHER_COLUMNS].copy()

anomaly_mask=(
    weather_daily.temperature_min_c.notna()
    & weather_daily.temperature_max_c.notna()
    & (weather_daily.temperature_min_c>weather_daily.temperature_max_c)
)
anomaly_count=int(anomaly_mask.sum())
print("Temperature-order anomalies:",anomaly_count)

if anomaly_count==3:
    old_min=weather_daily.loc[anomaly_mask,"temperature_min_c"].copy()
    weather_daily.loc[anomaly_mask,"temperature_min_c"]=weather_daily.loc[
        anomaly_mask,"temperature_max_c"
    ].to_numpy()
    weather_daily.loc[anomaly_mask,"temperature_max_c"]=old_min.to_numpy()
    weather_daily.loc[anomaly_mask,"temperature_mean_c"]=weather_daily.loc[
        anomaly_mask,["temperature_min_c","temperature_max_c"]
    ].mean(axis=1)
elif anomaly_count!=0:
    raise ValueError(f"Expected 0 or the 3 identified anomalies; found {anomaly_count}")

# Recompute the daily mean from the final Tmin/Tmax values.
# This also refreshes legacy cached Parquet files created before the skipna=False fix.
weather_daily["temperature_mean_c"] = (
    weather_daily[["temperature_min_c", "temperature_max_c"]]
    .mean(axis=1, skipna=False)
)

weather_daily.to_parquet(
    weather_daily_path,index=False,compression="snappy"
)

daily_checks=pd.DataFrame({
    "check":[
        "rows","stations","duplicate_station_date","missing_date",
        "negative_precipitation","temperature_min_gt_max","temperature_mean_mismatch",
        "missing_temperature_min","missing_temperature_max",
        "missing_precipitation"
    ],
    "value":[
        len(weather_daily),weather_daily.station_id.nunique(),
        weather_daily[["station_id","observation_date"]].duplicated().sum(),
        weather_daily.observation_date.isna().sum(),
        (weather_daily.precipitation_mm<0).sum(),
        (weather_daily.temperature_min_c>weather_daily.temperature_max_c).sum(),
        (
            (
                weather_daily["temperature_mean_c"]
                - weather_daily[["temperature_min_c","temperature_max_c"]].mean(axis=1, skipna=False)
            ).abs() > 1e-9
        ).fillna(False).sum(),
        weather_daily.temperature_min_c.isna().sum(),
        weather_daily.temperature_max_c.isna().sum(),
        weather_daily.precipitation_mm.isna().sum()
    ]
})
display(daily_checks)
assert daily_checks.loc[daily_checks["check"]=="duplicate_station_date","value"].iloc[0]==0
assert daily_checks.loc[daily_checks["check"]=="temperature_min_gt_max","value"].iloc[0]==0
assert daily_checks.loc[daily_checks["check"]=="temperature_mean_mismatch","value"].iloc[0]==0
assert daily_checks.loc[daily_checks["check"]=="negative_precipitation","value"].iloc[0]==0
print("Daily weather layer: PASS")

Temperature-order anomalies: 3


,check,value
0,rows,17680265
1,stations,1200
2,duplicate_station_date,0
3,missing_date,0
4,negative_precipitation,0
5,temperature_min_gt_max,0
6,temperature_mean_mismatch,0
7,missing_temperature_min,1566758
8,missing_temperature_max,1575462
9,missing_precipitation,1467326


Daily weather layer: PASS


## 10. Station-Year Weather Features

Create annual weather indicators from the validated daily layer.

In [20]:
weather_daily["year"]=weather_daily.observation_date.dt.year

# Only observation years from 1901 onward are relevant for the bridge study.
# Keep the raw DWD archive untouched, but exclude pre-1901 observations from all feature layers.
pre_1901_rows = int((weather_daily["year"] < MIN_WEATHER_YEAR).sum())
weather_daily = weather_daily.loc[weather_daily["year"] >= MIN_WEATHER_YEAR].copy()

print(f"Pre-{MIN_WEATHER_YEAR} daily observations excluded from feature engineering: {pre_1901_rows:,}")
assert weather_daily["year"].min() >= MIN_WEATHER_YEAR

weather_station_annual=(
    weather_daily.groupby(["station_id","year"],as_index=False)
    .agg(
        observation_days=("observation_date","nunique"),
        temperature_observation_days=("temperature_mean_c","count"),
        precipitation_observation_days=("precipitation_mm","count"),
        temperature_mean_c=("temperature_mean_c","mean"),
        temperature_min_c=("temperature_min_c","min"),
        temperature_max_c=("temperature_max_c","max"),
        precipitation_total_mm=("precipitation_mm",lambda s:s.sum(min_count=1)),
        precipitation_max_daily_mm=("precipitation_mm","max")
    )
)

frost=(
    weather_daily[weather_daily.temperature_min_c.notna()]
    .assign(frost_day=lambda d:(d.temperature_min_c<0).astype(int))
    .groupby(["station_id","year"],as_index=False)["frost_day"].sum()
    .rename(columns={"frost_day":"frost_days"})
)
hot=(
    weather_daily[weather_daily.temperature_max_c.notna()]
    .assign(hot_day=lambda d:(d.temperature_max_c>=30).astype(int))
    .groupby(["station_id","year"],as_index=False)["hot_day"].sum()
    .rename(columns={"hot_day":"hot_days"})
)
precip_days=(
    weather_daily[weather_daily.precipitation_mm.notna()]
    .assign(precip_day=lambda d:(d.precipitation_mm>0).astype(int))
    .groupby(["station_id","year"],as_index=False)["precip_day"].sum()
    .rename(columns={"precip_day":"precipitation_days"})
)

weather_station_annual=(
    weather_station_annual
    .merge(frost,on=["station_id","year"],how="left")
    .merge(hot,on=["station_id","year"],how="left")
    .merge(precip_days,on=["station_id","year"],how="left")
)
for c in ["frost_days","hot_days","precipitation_days"]:
    weather_station_annual[c]=weather_station_annual[c].fillna(0).astype(int)

weather_station_annual.to_parquet(
    weather_station_annual_path,index=False,compression="snappy"
)
weather_station_annual.to_sql(
    "dwd_weather_station_annual",engine,schema="transformed",
    if_exists="replace",index=False
)
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_dwd_weather_station_annual_station_year
        ON transformed.dwd_weather_station_annual(station_id,year)
    """))
display(weather_station_annual.head())
print("Station-year rows:",f"{len(weather_station_annual):,}")

Pre-1901 daily observations excluded from feature engineering: 483,109


,station_id,year,observation_days,temperature_observation_days,precipitation_observation_days,temperature_mean_c,temperature_min_c,temperature_max_c,precipitation_total_mm,precipitation_max_daily_mm,frost_days,hot_days,precipitation_days
0,00001,1937,365,365,365,9.000274,-11.0,32.8,817.4,39.1,89,7,166
1,00001,1938,365,365,365,8.675342,-14.0,31.5,766.5,30.2,101,5,148
2,00001,1939,365,365,365,8.265479,-21.2,29.4,1072.3,36.2,97,0,194
3,00001,1940,366,366,366,7.385929,-20.4,28.8,966.2,38.7,124,0,159
4,00001,1941,365,365,365,7.843288,-18.0,33.6,636.0,24.1,120,10,143


Station-year rows: 48,004


## 11. Bridge-Year Weather Features

Transfer the station-year features to the bridge level. The output is one record per **bridge-year** and retains station distance and mapping quality.

In [21]:
with engine.begin() as conn:
    conn.execute(text(f"""
        DROP TABLE IF EXISTS {BRIDGE_WEATHER_ANNUAL};
        CREATE TABLE {BRIDGE_WEATHER_ANNUAL} AS
        SELECT
            b.id_nr,b.bwnr,b.tbwnr,b.baujahr,
            m.weather_station_id,
            m.weather_source,
            m.weather_distance_km,
            m.weather_quality,
            w.year,
            w.observation_days,
            w.temperature_observation_days,
            w.precipitation_observation_days,
            w.temperature_mean_c,
            w.temperature_min_c,
            w.temperature_max_c,
            w.precipitation_total_mm,
            w.precipitation_max_daily_mm,
            w.frost_days,
            w.hot_days,
            w.precipitation_days
        FROM {BRIDGE_TABLE} b
        INNER JOIN {BRIDGE_WEATHER_MAP} m ON b.id_nr=m.id_nr
        INNER JOIN {STATION_ANNUAL} w ON m.weather_station_id=w.station_id
        WHERE b.baujahr>{MIN_BUILD_YEAR}
          AND b.baujahr IS NOT NULL
          AND b.id_nr IS NOT NULL
          AND b.geom_x IS NOT NULL
          AND b.geom_y IS NOT NULL
    """))
    conn.execute(text(f"""
        CREATE INDEX idx_bridge_weather_annual_bridge_year
        ON {BRIDGE_WEATHER_ANNUAL}(id_nr,year)
    """))
    conn.execute(text(f"""
        CREATE INDEX idx_bridge_weather_annual_station_year
        ON {BRIDGE_WEATHER_ANNUAL}(weather_station_id,year)
    """))

bridge_weather_annual=pd.read_sql(
    text(f"SELECT * FROM {BRIDGE_WEATHER_ANNUAL}"),engine
)
print("Bridge-year rows:",f"{len(bridge_weather_annual):,}")
print("Bridges:",f"{bridge_weather_annual.id_nr.nunique():,}")
display(bridge_weather_annual.head())

Bridge-year rows: 2,337,980
Bridges: 51,382


,id_nr,bwnr,tbwnr,baujahr,weather_station_id,weather_source,weather_distance_km,weather_quality,year,observation_days,temperature_observation_days,precipitation_observation_days,temperature_mean_c,temperature_min_c,temperature_max_c,precipitation_total_mm,precipitation_max_daily_mm,frost_days,hot_days,precipitation_days
0,8119557._1,8119557,1,1979,00001,DWD,3.538710,excellent,1937,365,365,365,9.000274,-11.0,32.8,817.4,39.1,89,7,166
1,8119557._2,8119557,2,1979,00001,DWD,3.538710,excellent,1937,365,365,365,9.000274,-11.0,32.8,817.4,39.1,89,7,166
2,8119559._1,8119559,1,1978,00001,DWD,3.507894,excellent,1937,365,365,365,9.000274,-11.0,32.8,817.4,39.1,89,7,166
3,8119559._2,8119559,2,1978,00001,DWD,3.507894,excellent,1937,365,365,365,9.000274,-11.0,32.8,817.4,39.1,89,7,166
4,8119560._0,8119560,0,1979,00001,DWD,3.530616,excellent,1937,365,365,365,9.000274,-11.0,32.8,817.4,39.1,89,7,166


## 12. Final Validation

Validate the complete chain from bridge population through station assignment and bridge-year weather features.

### QC clarification: annual temperature mean

The annual `temperature_mean_c` is the mean of daily mean temperatures. It must **not** be compared with `(annual Tmin + annual Tmax) / 2`; that midpoint is not generally equal to the annual mean. The correct annual QC is that the annual mean lies within the annual minimum/maximum range. Daily-level mean consistency is checked separately.


In [22]:
validation=pd.read_sql(
    text(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT id_nr) AS bridges,
            COUNT(DISTINCT weather_station_id) AS weather_stations,
            MIN(year) AS first_year,
            MAX(year) AS last_year,
            COUNT(*) FILTER(WHERE observation_days>0) AS rows_with_observations,
            COUNT(*) FILTER(WHERE temperature_observation_days>0) AS rows_with_temperature,
            COUNT(*) FILTER(WHERE precipitation_observation_days>0) AS rows_with_precipitation,
            COUNT(*) FILTER(WHERE weather_distance_km<0) AS negative_distances,
            COUNT(*) FILTER(WHERE weather_distance_km>{WEATHER_MAX_DISTANCE_KM}) AS over_distance_limit,
            COUNT(*) FILTER(WHERE baujahr<={MIN_BUILD_YEAR} OR baujahr IS NULL) AS invalid_bridge_year,
            COUNT(*) FILTER(WHERE year<{MIN_WEATHER_YEAR}) AS pre_1901_weather_rows,
            COUNT(*) FILTER(
                WHERE temperature_mean_c IS NOT NULL
                  AND temperature_min_c IS NOT NULL
                  AND temperature_mean_c < temperature_min_c
            ) AS temperature_mean_below_min,
            COUNT(*) FILTER(
                WHERE temperature_mean_c IS NOT NULL
                  AND temperature_max_c IS NOT NULL
                  AND temperature_mean_c > temperature_max_c
            ) AS temperature_mean_above_max,
            COUNT(*) FILTER(
                WHERE temperature_min_c IS NOT NULL
                  AND temperature_max_c IS NOT NULL
                  AND temperature_min_c > temperature_max_c
            ) AS temperature_order_error
        FROM {BRIDGE_WEATHER_ANNUAL}
    """),engine
)
display(validation)
v=validation.iloc[0]
assert int(v.negative_distances)==0
assert int(v.invalid_bridge_year)==0
assert int(v.pre_1901_weather_rows)==0
assert int(v.temperature_mean_below_min)==0
assert int(v.temperature_mean_above_max)==0
assert int(v.temperature_order_error)==0
assert int(v.first_year) >= MIN_WEATHER_YEAR
print("Bridge-year weather transformation: PASS")

,rows,bridges,weather_stations,first_year,last_year,rows_with_observations,rows_with_temperature,rows_with_precipitation,negative_distances,over_distance_limit,invalid_bridge_year,pre_1901_weather_rows,temperature_mean_below_min,temperature_mean_above_max,temperature_order_error
0,2337980,51382,1200,1901,2026,2337980,2176236,2197845,0,0,0,0,0,0,0


Bridge-year weather transformation: PASS


## 13. Visualization

## 12.1 QC Decision

**QC result:** PASS for the Weather feature pipeline after the checks in this notebook.

- Raw DWD station metadata is preserved before cleaning.
- Station IDs are unique in the cleaned station layer.
- Coordinates are validated and stored as EPSG:4326 POINT geometry.
- Bridge-to-station matching is one-to-one at bridge level.
- DWD station archive coverage is checked against mapped stations.
- Daily records are unique by `station_id + observation_date`.
- Negative precipitation and implausible temperatures are converted to missing values.
- The three known temperature-order anomalies are corrected before aggregation.
- Daily mean temperature is calculated only when both minimum and maximum temperature are available.
- Station-year and bridge-year keys are validated before promotion.
- Distances above 50 km are retained but explicitly classified as `too_far`; they are not silently deleted.


## 13. Promote Weather Features to `final.weather`

Promote the validated bridge-year weather feature table to the canonical `final` schema for the downstream ML dataset.

**Source:** `transformed.bridge_weather_annual`  
**Final:** `final.weather`

The transformed source remains available for reproducibility.

Only weather observation years `>= 1901` are promoted to the final feature layer; the raw DWD archive is not deleted.

In [23]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS final"))
    conn.execute(text(f"DROP TABLE IF EXISTS {FINAL_WEATHER_TABLE}"))
    conn.execute(text(f"""
        CREATE TABLE {FINAL_WEATHER_TABLE} AS
        SELECT *
        FROM {BRIDGE_WEATHER_ANNUAL}
        WHERE baujahr > {MIN_BUILD_YEAR}
          AND baujahr IS NOT NULL
          AND year >= {MIN_WEATHER_YEAR}
    """))
    conn.execute(text(f"""
        CREATE UNIQUE INDEX idx_final_weather_bridge_year
        ON {FINAL_WEATHER_TABLE}(id_nr, year)
    """))
    conn.execute(text(f"""
        CREATE INDEX idx_final_weather_station_year
        ON {FINAL_WEATHER_TABLE}(weather_station_id, year)
    """))

final_weather_check = pd.read_sql(
    text(f"""
        SELECT
            COUNT(*) AS bridge_year_rows,
            COUNT(DISTINCT id_nr) AS bridges,
            COUNT(DISTINCT weather_station_id) AS weather_stations,
            MIN(year) AS first_weather_year,
            MAX(year) AS last_weather_year,
            MIN(baujahr) AS min_baujahr,
            MAX(baujahr) AS max_baujahr
        FROM {FINAL_WEATHER_TABLE}
    """),
    engine
)
display(final_weather_check)

fw = final_weather_check.iloc[0]
assert int(fw["bridge_year_rows"]) > 0
assert int(fw["min_baujahr"]) > MIN_BUILD_YEAR
assert int(fw["first_weather_year"]) >= MIN_WEATHER_YEAR

duplicate_check = pd.read_sql(
    text(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT id_nr, year
            FROM {FINAL_WEATHER_TABLE}
            GROUP BY id_nr, year
            HAVING COUNT(*) > 1
        ) d
    """),
    engine
).iloc[0, 0]

assert int(duplicate_check) == 0
print(f"Final weather table created: {FINAL_WEATHER_TABLE}")
print(f"Bridge-year rows: {int(fw['bridge_year_rows']):,}")
print(f"Bridges: {int(fw['bridges']):,}")
print("PASS: final.weather is ready for downstream ML merging.")

,bridge_year_rows,bridges,weather_stations,first_weather_year,last_weather_year,min_baujahr,max_baujahr
0,2337980,51382,1200,1901,2026,1901,3019


Final weather table created: final.weather
Bridge-year rows: 2,337,980
Bridges: 51,382
PASS: final.weather is ready for downstream ML merging.


## 14. Final Results / Brief

In [24]:
result=pd.read_sql(
    text(f"""
        SELECT
            COUNT(DISTINCT id_nr) AS bridges,
            COUNT(*) AS bridge_year_rows,
            COUNT(DISTINCT weather_station_id) AS weather_stations,
            MIN(year) AS first_weather_year,
            MAX(year) AS last_weather_year,
            ROUND(AVG(weather_distance_km)::numeric,2) AS avg_station_distance_km,
            ROUND(MAX(weather_distance_km)::numeric,2) AS max_station_distance_km,
            COUNT(*) FILTER(
                WHERE weather_distance_km<={WEATHER_MAX_DISTANCE_KM}
            ) AS rows_within_distance_control
        FROM {FINAL_WEATHER_TABLE}
    """),engine
)
display(result)

r=result.iloc[0]
coverage=int(r.rows_within_distance_control)/int(r.bridge_year_rows)*100 if int(r.bridge_year_rows) else 0

print("="*70)
print("FINAL WEATHER FEATURE BRIEF")
print("="*70)
print(f"Bridges: {int(r.bridges):,}")
print(f"Bridge-year rows: {int(r.bridge_year_rows):,}")
print(f"DWD stations used: {int(r.weather_stations):,}")
print(f"Weather years: {int(r.first_weather_year)}–{int(r.last_weather_year)}")
print(f"Weather year filter: >= {MIN_WEATHER_YEAR}")
print(f"Average station distance: {r.avg_station_distance_km} km")
print(f"Maximum station distance: {r.max_station_distance_km} km")
print(f"Rows within distance control: {coverage:.2f}%")
print("\nOutput: final.weather")
print("\nWeather features:")
for x in [
    "observation_days","temperature_observation_days",
    "precipitation_observation_days","temperature_mean_c",
    "temperature_min_c","temperature_max_c",
    "precipitation_total_mm","precipitation_max_daily_mm",
    "frost_days","hot_days","precipitation_days"
]:
    print(" -",x)

,bridges,bridge_year_rows,weather_stations,first_weather_year,last_weather_year,avg_station_distance_km,max_station_distance_km,rows_within_distance_control
0,51382,2337980,1200,1901,2026,7.86,33.46,2337980


FINAL WEATHER FEATURE BRIEF
Bridges: 51,382
Bridge-year rows: 2,337,980
DWD stations used: 1,200
Weather years: 1901–2026
Weather year filter: >= 1901
Average station distance: 7.86 km
Maximum station distance: 33.46 km
Rows within distance control: 100.00%

Output: final.weather

Weather features:
 - observation_days
 - temperature_observation_days
 - precipitation_observation_days
 - temperature_mean_c
 - temperature_min_c
 - temperature_max_c
 - precipitation_total_mm
 - precipitation_max_daily_mm
 - frost_days
 - hot_days
 - precipitation_days


# Final Status

The reusable output is:

`final.weather` (source retained as `transformed.bridge_weather_annual`)

It is at **bridge-year level** and is ready for the downstream merge with the Bridge and Traffic feature layers.

### Final quality gates
- `Baujahr > 1900`
- Weather observation year `>= 1901`
- one DWD station per eligible bridge
- station distance calculated and classified; >50 km retained as an explicit QC flag
- DWD daily data cached in Parquet
- daily station/date uniqueness checked
- temperature, precipitation, and derived-mean consistency checked
- station-year features generated
- bridge-year weather features generated
- final bridge-year validation completed

**No prediction is performed.**